# 🔌 AI Agents + MCP — echter Server, echtes Protokoll

Im Notebook *„AI Agents under the Hood"* haben wir Tools selbst als Python-Funktionen gebaut. Hier läuft **derselbe Agent**, aber die Tools kommen über das **Model Context Protocol (MCP)** von einem **echten, separaten Serverprozess** — Kommunikation per **JSON-RPC über stdio**, genau wie bei Claude Desktop, Cursor & Co.

**Der rote Faden bleibt:** Es ist *derselbe* Agentic Loop. Nur die Herkunft der Tools ändert sich:

| Vorher (from scratch) | Jetzt (MCP) |
|---|---|
| Tool = lokale Python-Funktion | Tool lebt in einem **separaten Serverprozess** |
| Schema von Hand geschrieben | Schema kommt per `list_tools()` **vom Server** |
| Wir rufen die Funktion direkt | Wir rufen `session.call_tool()` **über das Protokoll** |

```
  Agent (MCP-Client)              MCP-Server (eigener Prozess)
       │  initialize()   ───────────►
       │  list_tools()   ───────────►
       │           ◄───────────  [Schemas]
       │  call_tool(name, args) ───►
       │           ◄───────────  [Ergebnis]
   gleicher Loop            deine Tools / Daten / APIs
```


## 0 · Setup

Wie im ersten Notebook: Konfiguration aus der `.env` (Azure OpenAI). Zusätzlich nötig: das offizielle **`mcp`**-SDK (steht in `pyproject.toml`, `uv sync`). Die Funktion `llm()` ist **identisch** — unser einziger Draht zum Modell.

In [9]:
import os, sys, json
from pathlib import Path
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
)
DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]

def llm(messages, tools=None, tool_choice="auto"):
    """Unser einziger Draht zum Modell - identisch zum ersten Notebook."""
    kwargs = dict(model=DEPLOYMENT, messages=messages)
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = tool_choice
    return client.chat.completions.create(**kwargs)

print("Client bereit. Deployment:", DEPLOYMENT)

Client bereit. Deployment: gpt-5.4-mini


## 1 · Der MCP-Server — ein echtes Programm in eigenem Prozess

Ein MCP-Server ist kein Notebook-Trick, sondern ein eigenständiges Programm. Wir schreiben es mit dem **offiziellen `mcp`-SDK** (FastMCP) in die Datei `mcp_demo_server.py`. Es stellt drei Tools bereit, die ein LLM allein *nicht* zuverlässig kann: die **Live-Serverzeit**, **Arithmetik** und eine **Wissensdatenbank** (deine Daten — nur auf dem Server).

Die nächste Zelle *schreibt* den Server als echte Datei (`%%writefile`) — gestartet wird er gleich als separater Prozess.

In [10]:
%%writefile mcp_demo_server.py
"""Ein echter MCP-Server (offizielles `mcp`-SDK, FastMCP) für den Vortrag.

Er spricht das echte Model Context Protocol (JSON-RPC 2.0 über stdio) und
stellt drei Tools bereit, die ein LLM allein NICHT zuverlässig kann:

  - current_time : Live-Serverzeit (kann das Modell nicht wissen)
  - add          : deterministische Arithmetik (zeigt sauber, dass das Tool lief)
  - search_kb    : durchsucht eine kleine Wissensdatenbank ("deine Daten via MCP")

Start (von Hand zum Ausprobieren):  uv run python mcp_demo_server.py
Im Notebook wird er als Subprozess über stdio gestartet — wie ein echter Client das tut.
"""
from datetime import datetime
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo-knowledge-server", log_level="WARNING")

# Eine winzige "Wissensdatenbank" – steht NUR auf dem Server, nicht im Modell.
KNOWLEDGE_BASE = {
    "mcp": (
        "Das Model Context Protocol (MCP) ist ein offener Standard von Anthropic (2024), "
        "der Tools, Ressourcen und Prompts über JSON-RPC zwischen Client und Server austauscht."
    ),
    "agent": (
        "Ein Agent ist ein LLM in einer Schleife mit Tools: fragen -> Tool ausführen -> "
        "Ergebnis anhängen -> erneut fragen, bis eine finale Antwort steht."
    ),
    "vortrag": (
        "Der Vortrag 'AI Agents under the Hood' baut einen Agenten from scratch und endet "
        "mit echtem MCP-Tool-Zugriff über einen separaten Serverprozess."
    ),
}


@mcp.tool()
def current_time() -> str:
    """Gibt das aktuelle Datum und die Uhrzeit des Servers im ISO-Format zurück."""
    return datetime.now().isoformat(timespec="seconds")


@mcp.tool()
def add(a: float, b: float) -> float:
    """Addiert zwei Zahlen und gibt die Summe zurück."""
    return a + b


@mcp.tool()
def search_kb(query: str) -> str:
    """Durchsucht die Wissensdatenbank des Servers nach einem Stichwort und gibt passende Einträge zurück."""
    q = query.lower()
    hits = [f"- {key}: {text}" for key, text in KNOWLEDGE_BASE.items()
            if q in key or q in text.lower()]
    return "\n".join(hits) if hits else f"Kein Eintrag zu '{query}' gefunden."


if __name__ == "__main__":
    # Standard-Transport von FastMCP ist stdio – genau das, was ein MCP-Client erwartet.
    mcp.run()


Overwriting mcp_demo_server.py


## 2 · Verbinden — mit Windows/Jupyter-Trick

Der Client startet den Server als **Subprozess** und spricht über stdio mit ihm.

> 🪟 **Wichtig auf Windows + Jupyter:** Jupyter läuft auf einem `SelectorEventLoop`, der **keine Subprozesse** starten kann (`NotImplementedError`), und Jupyters `sys.stderr` hat kein echtes `fileno()`. Lösung: Wir führen die MCP-Arbeit in einem **eigenen Thread mit eigenem `ProactorEventLoop`** aus (`run_async`) und leiten den Server-`stderr` nach `os.devnull`. Unter macOS/Linux ist das nicht nötig, schadet aber nicht.

In [11]:
import asyncio, threading
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# So startet ein MCP-Client einen Server: ein Befehl + Argumente. Mehr nicht.
SERVER = StdioServerParameters(command=sys.executable, args=["mcp_demo_server.py"])

# Server-stderr in den Nullspeicher leiten (Jupyters sys.stderr hat kein echtes fileno()).
_ERRLOG = open(os.devnull, "w")

def run_async(coro_factory):
    """Fuehrt eine async-Funktion in einem eigenen Event-Loop in einem eigenen Thread aus.
    Auf Windows ein ProactorEventLoop, damit Subprozesse (= der MCP-Server) starten koennen -
    unabhaengig von Jupyters SelectorEventLoop im Hauptthread."""
    box = {}
    def worker():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            box["v"] = loop.run_until_complete(coro_factory())
        except BaseException as e:
            box["e"] = e
        finally:
            loop.close()
    t = threading.Thread(target=worker, daemon=True)
    t.start(); t.join()
    if "e" in box:
        raise box["e"]
    return box["v"]

# Tools vom Server abfragen - die Schemas kommen NICHT von uns, sondern vom Server.
async def _list_tools():
    async with stdio_client(SERVER, errlog=_ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()          # Protokoll-Handshake
            return (await session.list_tools()).tools

server_tools = run_async(_list_tools)
for t in server_tools:
    print(f"🔌 {t.name}: {t.description}")

add_tool = next(t for t in server_tools if t.name == "add")
print("\nBeispiel-Schema (add):", json.dumps(add_tool.inputSchema, ensure_ascii=False))

🔌 current_time: Gibt das aktuelle Datum und die Uhrzeit des Servers im ISO-Format zurück.
🔌 add: Addiert zwei Zahlen und gibt die Summe zurück.
🔌 search_kb: Durchsucht die Wissensdatenbank des Servers nach einem Stichwort und gibt passende Einträge zurück.

Beispiel-Schema (add): {"properties": {"a": {"title": "A", "type": "number"}, "b": {"title": "B", "type": "number"}}, "required": ["a", "b"], "title": "addArguments", "type": "object"}


## 3 · MCP-Schema → OpenAI-Tool-Format

Das `inputSchema` eines MCP-Tools **ist** bereits ein JSON-Schema — es passt direkt in das `parameters`-Feld, das die OpenAI-/Azure-API erwartet. Eine triviale Umwandlung, kein Mapping-Aufwand. Genau dafür gibt es einen Standard.

In [12]:
def to_openai_tools(mcp_tools):
    return [{"type": "function", "function": {
        "name": t.name,
        "description": t.description or "",
        "parameters": t.inputSchema or {"type": "object", "properties": {}},
    }} for t in mcp_tools]

OPENAI_TOOLS = to_openai_tools(server_tools)
print(json.dumps(next(x for x in OPENAI_TOOLS if x["function"]["name"] == "add"),
                 indent=2, ensure_ascii=False))

{
  "type": "function",
  "function": {
    "name": "add",
    "description": "Addiert zwei Zahlen und gibt die Summe zurück.",
    "parameters": {
      "properties": {
        "a": {
          "title": "A",
          "type": "number"
        },
        "b": {
          "title": "B",
          "type": "number"
        }
      },
      "required": [
        "a",
        "b"
      ],
      "title": "addArguments",
      "type": "object"
    }
  }
}


## 4 · Der Agent — derselbe Loop, Tools über MCP

Jetzt der Kern: **exakt die Schleife** aus dem ersten Notebook. Nur zwei Stellen sind anders — markiert mit `# <- MCP`:
- die Tool-Schemas kommen per `list_tools()` vom Server,
- die Ausführung läuft über `await session.call_tool(name, args)` statt eines lokalen Funktionsaufrufs.

Der ganze Lauf passiert in einer offenen MCP-Session (im `run_async`-Thread) und wird danach sauber geschlossen.

In [13]:
def _mcp_text(result):
    """MCP-Tool-Ergebnis -> Text fuers Modell."""
    parts = [c.text for c in result.content if getattr(c, "type", None) == "text"]
    return "\n".join(parts) if parts else str(result.content)

async def _run_mcp_agent(task, max_steps=8, verbose=True):
    async with stdio_client(SERVER, errlog=_ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = to_openai_tools((await session.list_tools()).tools)   # <- MCP: Schemas vom Server

            messages = [{"role": "user", "content": task}]
            for step in range(1, max_steps + 1):
                msg = llm(messages, tools=tools).choices[0].message

                a = {"role": "assistant", "content": msg.content}
                if msg.tool_calls:
                    a["tool_calls"] = [{"id": tc.id, "type": "function",
                        "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                        for tc in msg.tool_calls]
                messages.append(a)

                if not msg.tool_calls:
                    if verbose: print(f"[Schritt {step}] ✓ finale Antwort")
                    return msg.content

                for tc in msg.tool_calls:
                    args = json.loads(tc.function.arguments or "{}")
                    if verbose: print(f"[Schritt {step}] → MCP call_tool {tc.function.name}({args})")
                    result = await session.call_tool(tc.function.name, args)   # <- MCP: Ausfuehrung ueber das Protokoll
                    text = _mcp_text(result)
                    if verbose: print(f"            ⤷ {text[:90]}")
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": text})
            return "(max_steps erreicht)"

def run_mcp_agent(task, **kw):
    """Synchroner Wrapper - startet den Agentenlauf im Proactor-Thread."""
    return run_async(lambda: _run_mcp_agent(task, **kw))

answer = run_mcp_agent(
    "Wie spaet ist es laut Server? Rechne ausserdem 17+25, "
    "und was weiss die Wissensdatenbank ueber MCP?"
)
print("\n=== Ergebnis ===\n", answer)

[Schritt 1] → MCP call_tool current_time({})
            ⤷ 2026-06-16T14:31:57
[Schritt 1] → MCP call_tool add({'a': 17, 'b': 25})
            ⤷ 42.0
[Schritt 1] → MCP call_tool search_kb({'query': 'MCP'})
            ⤷ - mcp: Das Model Context Protocol (MCP) ist ein offener Standard von Anthropic (2024), der
[Schritt 2] ✓ finale Antwort

=== Ergebnis ===
 Laut Server ist es: **2026-06-16T14:31:57**

**17 + 25 = 42**

Zur Wissensdatenbank über **MCP**:
- **MCP** steht für **Model Context Protocol**.
- Es ist ein **offener Standard von Anthropic (2024)**.
- Er dient dazu, **Tools, Ressourcen und Prompts über JSON-RPC zwischen Client und Server auszutauschen**.
- Außerdem gibt es einen Eintrag zu einem Vortrag **„AI Agents under the Hood“**, der einen Agenten from scratch baut und mit echtem **MCP-Tool-Zugriff** über einen separaten Serverprozess endet.


👉 Drei MCP-Tool-Aufrufe, eine saubere Antwort — und der Loop ist Zeile für Zeile derselbe wie im ersten Notebook. **Das ist der ganze Wert von MCP:** dieselbe Agentenschleife, aber die Werkzeuge stecken in austauschbaren Servern.

## 5 · Interaktiv — frag den Agenten selbst (mit Gedächtnis)

Jetzt **du**: Die Zelle fragt in einer Schleife nach einer Eingabe und schickt sie an den MCP-Agenten.

**Neu — Gedächtnis (Rückbezug auf Kapitel 2):** Wir führen eine Historie `chat_messages`, die über *alle* Fragen hinweg wächst. Dadurch funktionieren **Folgefragen** wie *„und addiere 10 dazu"* — der Agent kennt das vorige Ergebnis. Live siehst du, welche **MCP-Tools** er aufruft.

> Probier eine Kette: *„Addiere 128 und 96"* → *„und das mal 2?"* → *„Was weiß die Wissensdatenbank über Agenten?"*
> `reset` löscht das Gedächtnis, leere Eingabe oder `exit` beendet.

In [ ]:
# Interaktiver MCP-Agent MIT Gedaechtnis: die Historie bleibt ueber Fragen hinweg erhalten.
chat_messages = []   # <- DAS ist das Gedaechtnis (Kapitel 2): waechst mit jeder Frage + Antwort

async def _chat_turn(messages, max_steps=8):
    """Beantwortet EINE Nutzerfrage - auf Basis der GESAMTEN bisherigen Historie."""
    async with stdio_client(SERVER, errlog=_ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = to_openai_tools((await session.list_tools()).tools)
            for step in range(1, max_steps + 1):
                msg = llm(messages, tools=tools).choices[0].message      # bekommt die GANZE Historie
                a = {"role": "assistant", "content": msg.content}
                if msg.tool_calls:
                    a["tool_calls"] = [{"id": tc.id, "type": "function",
                        "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                        for tc in msg.tool_calls]
                messages.append(a)                                       # Antwort ins Gedaechtnis
                if not msg.tool_calls:
                    return msg.content
                for tc in msg.tool_calls:
                    args = json.loads(tc.function.arguments or "{}")
                    print(f"   → MCP call_tool {tc.function.name}({args})")
                    result = await session.call_tool(tc.function.name, args)
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": _mcp_text(result)})
            return "(max_steps erreicht)"

print("MCP-Agent mit Gedaechtnis bereit. Folgefragen funktionieren!")
print("('reset' loescht das Gedaechtnis, leere Eingabe oder 'exit' beendet.)\n")

while True:
    frage = input("🙋 Deine Frage (oder 'exit'): ").strip()
    if not frage or frage.lower() in ("exit", "quit", "ende"):
        print("Beendet.")
        break
    if frage.lower() == "reset":
        chat_messages.clear()
        print("Gedaechtnis geloescht.\n")
        continue
    chat_messages.append({"role": "user", "content": frage})             # Frage ins Gedaechtnis
    antwort = run_async(lambda: _chat_turn(chat_messages))               # gleicher Proactor-Thread-Trick
    print("\n🤖", antwort, "\n")

## 6 · (Optional) Ein echter Drittanbieter-Server

Der Beweis, dass es ein *Standard* ist: Wir tauschen nur den **Server-Befehl** aus — der Client-Code (inkl. `run_async`) bleibt identisch. Hier der offizielle **Filesystem-Server** von Anthropic (läuft über `npx`, lädt beim ersten Start das npm-Paket; braucht Node.js).

> ⚠️ Braucht Internet beim ersten Start (npm-Download) und Node.js. Schlägt es fehl, ist das für den Vortrag egal — der eigene Server oben ist der Kern.

In [ ]:
WS = Path("./agent_workspace").resolve()
WS.mkdir(exist_ok=True)
(WS / "hallo.txt").write_text("Hallo vom Filesystem-MCP-Server!", encoding="utf-8")

# Exakt derselbe Client-Code wie oben - nur ein anderer Server-Befehl:
FS_SERVER = StdioServerParameters(
    command="npx.cmd",   # Windows: .cmd  |  macOS/Linux: "npx"
    args=["-y", "@modelcontextprotocol/server-filesystem", str(WS)],
)

async def _fs_demo():
    async with stdio_client(FS_SERVER, errlog=_ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            names = [t.name for t in (await session.list_tools()).tools]
            print("Tools des Filesystem-Servers:", names)
            res = await session.call_tool("list_directory", {"path": str(WS)})
            print("\nlist_directory:\n", _mcp_text(res))

try:
    run_async(_fs_demo)
except Exception as e:
    print("Optionaler Server nicht verfuegbar (Node/npx/Netz?):", type(e).__name__, e)

## Mitnehmen

1. **MCP ändert nichts am Agenten.** Derselbe Loop — die Tools kommen nur aus einem anderen Prozess.
2. **Discovery statt Hardcoding:** `list_tools()` liefert die Schemas, `call_tool()` führt aus — über JSON-RPC/stdio.
3. **Ein Server, viele Clients:** denselben Server nutzen Claude Desktop, Cursor, deine App — *und* der selbstgebaute Agent aus dem ersten Notebook.
4. **Windows-Detail:** MCP-Subprozesse brauchen in Jupyter einen `ProactorEventLoop` im Worker-Thread (`run_async`).